# Анализ СТП

`input -> document_parser (+ RAG) -> formatter -> output `

In [22]:
from src import Store

FILE_PATH = './data/documents/2. Региональный уровень/СТП Отходы.docx'

store = Store(FILE_PATH)

In [23]:
base_retriever = store._store.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k': 20,
        'filter': store._get_filter({})
    }
)

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever

parent_retriever = ParentDocumentRetriever(
    vectorstore=store._store,
    docstore=docstore,
    child_splitter=child_splitter,   # как вы уже чанковали
    parent_splitter=parent_splitter, # например, по страницам или разделам
)

In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from src import llm

final_retriever = ContextualCompressionRetriever(
base_retriever=parent_retriever,
base_compressor=LLMChainExtractor.from_llm(llm)
)

In [26]:
final_retriever.invoke('Цели')

[Document(metadata={'page_number': 1, 'category': 'TableChunk', 'element_id': '190a6e376ffa75bf563d4aaacf35f802', 'chunk_index': 1}, page_content='Назначение Обработка, утилизация и размещение твердых коммунальных отходов III – V классов опасности и отдельных видов промышленных отходов  \nНазначение Обработка, утилизация и размещение твердых коммунальных отходов III – V классов опасности и отдельных видов промышленных отходов'),
 Document(metadata={'page_number': 1, 'is_continuation': True, 'category': 'TableChunk', 'element_id': '923abe83aed3c598d486166c64873e88', 'chunk_index': 2}, page_content='Назначение Обработка, утилизация, обезвреживание и размещение твердых коммунальных отходов IV и V классов опасности и отдельных видов промышленных отходов  \nНазначение Обработка, утилизация и размещение твердых коммунальных отходов III – V классов опасности и отдельных видов промышленных отходов'),
 Document(metadata={'page_number': 1, 'is_continuation': True, 'category': 'TableChunk', 'elem

In [3]:
from src import Model
from pydantic import Field

class QueryModel(Model):
    """
    Информация о запросе пользователя
    """
    settlement : str = Field(description='Населенный пункт')
    location : str = Field(description='Расположение')
    current_year : int = Field(description='Текущий год')

query = QueryModel(
    settlement='г. Гатчина',
    location='Северо-западный федеральный округ, Ленинградская область, Гатчинский муниципальный округ (бывш. Гатчинский муниципальный район)',
    current_year=2026,
)

In [4]:
class ObjectModel(Model):
    """
    Описание объекта
    """
    name : str = Field(description='Наименование объекта')
    location : str = Field(description='Расположение объекта')
    params : str = Field(description='Характеристики объекта')

class SchemaModel(Model):
    """
    Результаты анализа документа схемы территориального планирования
    """
    objects : list[ObjectModel] = Field(min_length=0, description='Перечень объектов, планируемых к размещению')

In [15]:
from src import Agent

agent = Agent(tools=[store.tool], debug=True, response_format=SchemaModel, system_prompt="""
Ты - аналитик, работающий строго с имеющимся в векторной базе разбитым на чанки документом.
    
ПРАВИЛА РАБОТЫ:
- ИСПОЛЬЗУЙ инструменты для работы с документом.
- Используй ТОЛЬКО информацию из чанков и conversation summary (если есть). НЕ ИСПОЛЬЗУЙ общие знания и НЕ ПРИДУМЫВАЙ информацию.
              
ПРАВИЛА ОСТАНОВКИ:
- Чанки ПОВТОРЯЮТСЯ между запросами, а новая информация незначительна.

ВАЖНО: Твой ответ ОБЯЗАН соответствовать структуре response_format.
              
ЗАПРОС ПОЛЬЗОВАТЕЛЯ:
Какие объекты относятся к: г. Гатчина, Гатчинский муниципальный район (округ)?
""")


res=agent.run([])

[values] {'messages': []}
[updates] {'SummarizationMiddleware.before_model': None}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 750, 'total_tokens': 933, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss:120b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-457', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cfc1d-e14c-7650-9ee5-61378ab7db7c-0', tool_calls=[{'name': 'search', 'args': {'filter': None, 'k': 10, 'query': 'г. Гатчина Гатчинский муниципальный район объекты', 'retriever': None, 'search_type': 'similarity'}, 'id': 'call_fgerquhk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 750, 'output_tokens': 183, 'total_tokens': 933, 'input_token_details': {}, 'output_token_details': {}})]}}
[values] {'messages': [AIMessage(co

KeyboardInterrupt: 

In [ ]:
res.model_dump()